# Tome 14, Chapter 2: The Selberg–Delange method (Python companion)

Exercises first, solutions at the end. We check Hankel's formula for $1/\Gamma(z)$ by numerical integration, test the Selberg–Delange asymptotics for $\sum_{n\le x}z^{\omega(n)}$ up to $10^7$, race Landau against Ramanujan on the sums of two squares, count $\pi_k(x)$, and compute the Landau–Ramanujan constant to fifteen digits. The computer science application: **accelerating slowly convergent Euler products** by Möbius inversion of $\log\zeta$ and $\log L$. Requires `numpy` and `mpmath`.

In [ ]:
import math, time
import numpy as np
import mpmath as mp

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

N = 10**7
P = primes_upto(N)
omega = np.zeros(N + 1, dtype=np.int8)
for p in P: omega[p::p] += 1
omega = omega[1:]            # omega[n-1] = omega(n)
print(len(P), "primes up to", N)

## Exercises

**P2.1 (★) Hankel's loop.** Integrate $\frac1{2\pi i}\int_{\mathcal H}w^{-z}e^w\,\mathrm dw$ numerically along the two edges and the unit circle (Lemma 2.1) for $z=\frac12$, $3$, $-1$ and $-1.5+2i$, and compare with $1/\Gamma(z)$.

**P2.2 (★) The main term.** For $z=\frac12,2,3$ compute the ratio of $\sum_{n\le x}z^{\omega(n)}$ to $x(\log x)^{z-1}F(z)/\Gamma(z)$ for $x=10^3,\dots,10^7$ (Figure 2.2). Theorem 2.5 says the ratio is $1+O(1/\log x)$: is $(\text{ratio}-1)\log x$ roughly constant?

**P2.3 (★) The vanishing main term.** Compute $\frac1x\sum_{n\le x}(-1)^{\omega(n)}$ for $x=10^4,\dots,10^7$ (Exercise 2.3) and compare with $(\log x)^{-3}$.

**P2.4 (★★) Landau versus Ramanujan.** Count the sums of two squares up to $10^4,\dots,10^7$ with a sieve (Tome 1, Theorem 7.3), and compare with Landau's $bx/\sqrt{\log x}$ and Ramanujan's $b\int_2^x\mathrm dt/\sqrt{\log t}$.

**P2.5 (★★) Primes $3$ modulo $4$ only.** Count the $n\le x$ all of whose prime factors are $\equiv3\pmod 4$, and compare with $c_3x/\sqrt{\log x}$ (Exercise 2.5).

**P2.6 (★★) Integers with $k$ prime factors.** For $x=10^7$ compare $\pi_k(x)$, $k=1,\dots,8$, with Landau's $\frac x{\log x}\frac{(\log\log x)^{k-1}}{(k-1)!}$ (Exercise 2.6) and with the Sathe–Selberg formula of Chapter 3, which multiplies Landau's expression by $F(r)/\Gamma(r+1)$ at $r=\frac{k-1}{\log\log x}$.

**P2.7 (★★★) Computer science: fifteen digits of $b$.** The product $\prod_{p\equiv3\,(4)}(1-p^{-2})^{-1/2}$ converges slowly: the tail after $p$ is about $1/(4p\log p)$. Use $\sum_p\chi(p)p^{-s}=\sum_{k\ge1}\frac{\mu(k)}k\log L(ks,\chi^k)$ (Möbius inversion of $\log L(s,\chi)=\sum_{k}\frac1k\sum_p\chi(p)^kp^{-ks}$) for $\chi=\chi_0,\chi_4$ modulo $4$ to compute $b$ to $15$ digits. How many primes would the naive product need?

## Solutions

### P2.1

In [ ]:
mp.mp.dps = 20
def hankel(z):
    z = mp.mpc(z)
    # lower edge: w = u e^{-i pi}, u from inf to 1 ; upper edge: w = u e^{i pi}, u from 1 to inf
    low = mp.quad(lambda u: mp.exp(-z*(mp.log(u) - 1j*mp.pi)) * mp.exp(-u) * (-1), [mp.inf, 1])
    up = mp.quad(lambda u: mp.exp(-z*(mp.log(u) + 1j*mp.pi)) * mp.exp(-u) * (-1), [1, mp.inf])
    circ = mp.quad(lambda th: mp.exp(-z*1j*th) * mp.exp(mp.expj(th)) * 1j*mp.expj(th), [-mp.pi, mp.pi])
    return (low + up + circ) / (2j*mp.pi)
for z in [0.5, 3, -1, mp.mpc(-1.5, 2)]:
    print(z, mp.nstr(hankel(z), 12), mp.nstr(mp.rgamma(z), 12))

The loop reproduces $1/\Gamma$ to working precision, including the zero at $z=-1$ and complex arguments.

### P2.2

In [ ]:
def F(z):
    # product over p <= N, plus the tail estimate exp(-(z-1)^2/2 * sum_{p>N} 1/p^2) ~ 1
    return float(np.exp(np.sum(np.log1p((z - 1) / P) + (z - 1) * np.log1p(-1 / P))))
xs = [10**k for k in range(3, 8)]
for z in [0.5, 2, 3]:
    S = np.cumsum(np.power(float(z), omega.astype(float)))
    row = []
    for x in xs:
        L = math.log(x)
        r = S[x - 1] / (x * L**(z - 1) * F(z) / math.gamma(z))
        row.append(f"{r:.4f} ({(r-1)*L:.2f})")
    print(f"z={z}: F={F(z):.5f}", "  ".join(row))
print("6/pi^2 =", 6 / math.pi**2)

In parentheses, $(\text{ratio}-1)\log x$. It is almost constant, about $1.29$ for $z=2$ and $7.3$ for $z=3$, as the expansion $1+\frac{c_1(z)}{\log x}+\frac{c_2(z)}{\log^2x}+\cdots$ of the Remark after Theorem 2.5 predicts; for $z=\frac12$ it is still drifting towards its limit. The large $c_1(3)$ is why the $z=3$ curve of Figure 2.2 is so far from $1$.

### P2.3

In [ ]:
S = np.cumsum(np.where(omega % 2 == 0, 1.0, -1.0))
for k in range(4, 8):
    x = 10**k
    print(x, S[x - 1] / x, " (log x)^-3 =", math.log(x)**-3)

The mean is tiny, and from $10^5$ on it falls much faster than any fixed power of $\log x$ would suggest over such a short range; at $10^7$ it is below $(\log x)^{-3}$. The main term carries the factor $1/\Gamma(-1)=0$, and so do all the terms of Delange's expansion.

### P2.4

In [ ]:
P3 = P[P % 4 == 3]
bad = np.zeros(N + 1, dtype=bool)      # bad[n]: some p = 3 mod 4 divides n to an odd power
r = int(N ** 0.5)
for p in P3:
    if p > r:
        bad[p::p] = True               # v_p(n) <= 1 for every n <= N
    else:
        par = np.zeros(N + 1, dtype=bool)
        pk = p
        while pk <= N:
            par[pk::pk] ^= True        # parity of v_p(n)
            pk *= p
        bad |= par
two_sq = ~bad; two_sq[0] = False
B = np.cumsum(two_sq)
b_naive = 2**-0.5 * math.exp(-0.5 * np.sum(np.log1p(-1.0 / P3.astype(float)**2)))
b = 0.764223653589220              # from P2.7
print("product over p <= 10^7 gives", b_naive)
for k in range(4, 8):
    x = 10**k; L = math.log(x)
    lan = b * x / math.sqrt(L)
    ram = b * float(mp.quad(lambda t: 1 / mp.sqrt(mp.log(t)), [2, 10, 100, x]))
    print(f"x=10^{k}: B={B[x]}, Landau {lan:.0f} ({B[x]/lan:.4f}), Ramanujan {ram:.0f} ({B[x]/ram:.4f})")

Ramanujan's integral is closer by a factor of about five at $10^7$: its relative error is under one percent, Landau's over four. Shanks (1964) proved that the integral wins for all large $x$, although its error is not $O(\sqrt x/\log x)$ as Ramanujan claimed.

### P2.5

In [ ]:
only3 = np.ones(N + 1, dtype=bool); only3[0] = False
for p in P[(P % 4 == 1) | (P == 2)]:
    only3[p::p] = False
C3 = np.cumsum(only3)
c3 = math.sqrt(2) / math.pi * (b * math.sqrt(2))   # sqrt(2)/pi * prod (1-p^-2)^(-1/2)
print("c3 =", c3)
for k in range(5, 8):
    x = 10**k
    print(x, C3[x], C3[x] / (c3 * x / math.sqrt(math.log(x))))

### P2.6

In [ ]:
x = N; L = math.log(x); lam = math.log(L)
counts = np.bincount(omega, minlength=10)
for k in range(1, 9):
    landau = x / L * lam**(k - 1) / math.factorial(k - 1)
    r = (k - 1) / lam
    ss = landau * F(r) / math.gamma(r + 1)
    print(f"k={k}: pi_k={counts[k]:8d}  Landau {landau:9.0f} ({counts[k]/landau:.3f})  Sathe-Selberg {ss:9.0f} ({counts[k]/ss:.3f})")

At $x=10^7$, $\log\log x\approx2.78$ is small and neither formula is accurate. Landau's is off by half for $k=2,3$ and by orders of magnitude for $k\ge6$. The Sathe–Selberg factor helps for most $k$ but still leaves errors of $15$ to $30$ percent, since its error term is of relative size $k/(\log\log x)^2$. For $k\ge7$ both fail completely: no $n\le10^7$ has more than $8$ prime factors, because $2\cdot3\cdots23>10^7$, and Chapter 3 proves the Sathe–Selberg formula only for $k\le(2-\varepsilon)\log\log x$. Probabilistic number theory is a theory of very large $x$.

### P2.7

In [ ]:
mp.mp.dps = 30
chi4 = [0, 1, 0, -1]
def L4(s):  return mp.dirichlet(s, chi4)
def L0(s):  return mp.zeta(s) * (1 - mp.power(2, -s))      # principal character mod 4
def mobius(k):
    m, res, d = k, 1, 2
    while d * d <= m:
        if m % d == 0:
            m //= d
            if m % d == 0: return 0
            res = -res
        d += 1
    return -res if m > 1 else res
def primesum(s, which):
    # sum over odd primes of chi(p) p^{-s}; chi = chi0 or chi4
    tot = mp.mpf(0)
    for k in range(1, 80):
        mu = mobius(k)
        if mu == 0: continue
        Lk = L0(k * s) if (which == 0 or k % 2 == 0) else L4(k * s)
        tot += mp.mpf(mu) / k * mp.log(Lk)
    return tot
def sum_p3(s):   # sum over p = 3 mod 4 of p^{-s}
    return (primesum(s, 0) - primesum(s, 4)) / 2
# log prod_{p=3(4)} (1-p^-2)^(-1/2) = 1/2 sum_m (1/m) sum_{p=3(4)} p^{-2m}
logprod = mp.mpf(0)
for m in range(1, 60):
    logprod += sum_p3(2 * m) / m / 2
b_acc = mp.exp(logprod) / mp.sqrt(2)
print("b =", mp.nstr(b_acc, 18))
print("naive product over p <= 10^7:", b_naive, " error", float(b_acc) - b_naive)

The accelerated value is $b=0.764223653589220\ldots$, while the product over the $3\cdot10^5$ primes $p\equiv3\pmod4$ below $10^7$ is wrong in the ninth digit. Since the tail after $p$ is about $\frac1{4p\log p}$, fifteen digits by brute force would need primes up to about $10^{13}$, while the Möbius trick needs a few dozen values of $\zeta$ and $L(s,\chi_4)$ at $s\ge2$, each computed in milliseconds. The same trick computes $F(z)$ of Corollary 2.7 for any $z$.